# RHI LoRA v3 Training — slot_builder_lora_v3 + shape_critic_v1
**Phase 1163+ | QuHarmonics Research Group**

## v8 → v9 Path

```
v8 PATCHED corpus (78 repair + 12 shape rows)
  ↓
Build training datasets
  ↓
Train slot_builder_lora_v3 (16 examples, 5 epochs)
Train shape_critic_v1 (3 examples, 8 epochs)
  ↓
v9 shape-guided synthesis runtime
```

$$\boxed{Q \rightarrow C_{\text{v3}} \rightarrow K^*_{\text{shape critic}} \rightarrow A_{\text{shape-guided}} \rightarrow \Psi/\Omega}$$

In [12]:
# ============================================================
# CONFIGURATION
# ============================================================
from pathlib import Path
import json

# Paths
ROOT = Path.cwd()
V8_OUTPUT_DIR = ROOT / "rhi_live_runtime_v8_outputs"
DATASET_DIR = ROOT / "rhi_lora_v3_training_data"
SLOT_OUTPUT_DIR = ROOT / "slot_builder_lora_v3"
SHAPE_OUTPUT_DIR = ROOT / "shape_critic_v1"

# Create directories
DATASET_DIR.mkdir(parents=True, exist_ok=True)
SLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHAPE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = False  # Set True if CUDA OOM
MAX_LENGTH = 2048

# Training flags
BUILD_DATASETS = True
RUN_TRAINING = True  # SET TO TRUE WHEN READY TO TRAIN

# slot_builder_lora_v3 hyperparameters
SLOT_LORA_RANK = 16
SLOT_LORA_ALPHA = 32
SLOT_LORA_DROPOUT = 0.05
SLOT_TARGET_MODULES = ["q_proj", "v_proj"]
SLOT_EPOCHS = 5
SLOT_LR = 3e-4
SLOT_BATCH_SIZE = 1
SLOT_GRAD_ACCUM = 4

# shape_critic_v1 hyperparameters
SHAPE_LORA_RANK = 8
SHAPE_LORA_ALPHA = 16
SHAPE_LORA_DROPOUT = 0.05
SHAPE_TARGET_MODULES = ["q_proj", "k_proj", "v_proj"]
SHAPE_EPOCHS = 8
SHAPE_LR = 5e-4
SHAPE_BATCH_SIZE = 1
SHAPE_GRAD_ACCUM = 4

print(f"ROOT: {ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"V8_OUTPUT_DIR: {V8_OUTPUT_DIR}")
print(f"\nRUN_TRAINING: {RUN_TRAINING}")
print(f"USE_4BIT: {USE_4BIT}")

ROOT: D:\Nexus\Nexus Mark 9\NoteBooks
DATASET_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data
V8_OUTPUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_live_runtime_v8_outputs

RUN_TRAINING: True
USE_4BIT: False


In [13]:
# ============================================================
# IMPORTS
# ============================================================
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import pandas as pd
from collections import defaultdict

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060


## Load v8 Corpus

In [14]:
# ============================================================
# LOAD V8 CORPUS
# ============================================================

# Load repair training rows
repair_file = V8_OUTPUT_DIR / "rhi_repair_training_rows_v8.jsonl"
with open(repair_file) as f:
    repair_rows = [json.loads(line) for line in f]

# Load shape training rows
shape_file = V8_OUTPUT_DIR / "rhi_shape_training_rows_v8.jsonl"
with open(shape_file) as f:
    shape_rows = [json.loads(line) for line in f]

# Load run data (to get contracts)
runs_file = V8_OUTPUT_DIR / "rhi_live_runs_v8.jsonl"
with open(runs_file) as f:
    runs = [json.loads(line) for line in f]

print(f"Loaded {len(repair_rows)} repair rows")
print(f"Loaded {len(shape_rows)} shape rows")
print(f"Loaded {len(runs)} runs")

# Create run_id -> contract mapping
run_contracts = {}
for run in runs:
    run_id = run['run_id']
    contract_result = run.get('contract_result', {})
    contract = contract_result.get('contract', {})
    run_contracts[run_id] = contract

print(f"\nExtracted {len(run_contracts)} contracts from runs")

Loaded 78 repair rows
Loaded 12 shape rows
Loaded 16 runs

Extracted 16 contracts from runs


## Build slot_builder_lora_v3 Dataset

In [15]:
# ============================================================
# BUILD SLOT_BUILDER TRAINING EXAMPLES
# ============================================================

SLOT_SYSTEM_PROMPT = """You are the Nexus Slot Constructor. Generate missing-shape contracts. Required fields: family_class, domain_carrier, forbidden_neighbor_carrier, boundary_conditions, preserved_function, failure_modes, witness_readout, residue. family_class must be a complete noun phrase. Use operational fit, not labels."""

SLOT_USER_TEMPLATE = """Prompt: {prompt}

Generate the missing-shape contract.
Checklist:
1. Need: occupy the inverse cavity.
2. Function: preserve or redirect the required operation.
3. Boundary: respect constraints.
4. Trap: reject noun/surface-label confusion.
5. Collapse: produce one executable witness/readout.
family_class must be a complete noun phrase (not a dangling preposition).
Return JSON only."""

def build_slot_training_examples(repair_rows, run_contracts):
    """Build slot_builder training examples from repair rows."""
    
    # Group repairs by run_id
    by_run = defaultdict(list)
    for row in repair_rows:
        by_run[row['run_id']].append(row)
    
    examples = []
    for run_id, repairs in by_run.items():
        # Get the repaired contract for this run
        contract = run_contracts.get(run_id)
        if not contract:
            print(f"Warning: No contract found for {run_id}")
            continue
        
        # Get prompt (same for all repairs in this run)
        prompt = repairs[0]['prompt']
        
        # Create training example
        example = {
            "messages": [
                {"role": "system", "content": SLOT_SYSTEM_PROMPT},
                {"role": "user", "content": SLOT_USER_TEMPLATE.format(prompt=prompt)},
                {"role": "assistant", "content": json.dumps(contract, ensure_ascii=False)}
            ]
        }
        examples.append(example)
    
    return examples

if BUILD_DATASETS:
    slot_examples = build_slot_training_examples(repair_rows, run_contracts)
    
    # Save training examples
    slot_train_file = DATASET_DIR / "slot_builder_lora_v3_train.jsonl"
    with open(slot_train_file, 'w') as f:
        for ex in slot_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    
    print(f"\nBuilt {len(slot_examples)} slot_builder examples")
    print(f"Saved to: {slot_train_file}")
    
    # Show example
    print("\nExample 1:")
    ex = slot_examples[0]
    print(f"  System: {ex['messages'][0]['content'][:80]}...")
    print(f"  User: {ex['messages'][1]['content'][:80]}...")
    print(f"  Assistant: {ex['messages'][2]['content'][:120]}...")


Built 16 slot_builder examples
Saved to: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data\slot_builder_lora_v3_train.jsonl

Example 1:
  System: You are the Nexus Slot Constructor. Generate missing-shape contracts. Required f...
  User: Prompt: Using the Nexus lens, explain why current AI agents fail when they use t...
  Assistant: {"family_class": "operational closure of nexus slot construction", "domain_carrier": ["AI", "agents", "fail", "tools", "...


## Build shape_critic_v1 Dataset

In [16]:
# ============================================================
# BUILD SHAPE_CRITIC TRAINING EXAMPLES
# ============================================================

SHAPE_SYSTEM_PROMPT = """You are the Nexus Shape Critic. Given a contract and prompt, predict the dominant operation-shape and detect composites. Output strict JSON with: dominant_shape, dominant_mass, composite, shape_field_mass, normalized_masses, confidence."""

SHAPE_USER_TEMPLATE = """Contract:
{contract}

Prompt: {prompt}

Predict the dominant operation-shape using shape-field mass across all branches."""

def build_shape_training_examples(shape_rows, run_contracts):
    """Build shape_critic training examples from shape rows.
    
    Each run has 4 shape rows (one per branch).
    We aggregate by run_id to create one training example per run.
    """
    
    # Group by run_id
    by_run = defaultdict(list)
    for row in shape_rows:
        by_run[row['run_id']].append(row)
    
    examples = []
    for run_id, rows in by_run.items():
        # All rows from same run have same contract, prompt, shape_field_mass, composite
        first = rows[0]
        
        # Get contract
        contract = run_contracts.get(run_id)
        if not contract:
            print(f"Warning: No contract found for {run_id}")
            continue
        
        prompt = first['prompt']
        
        # Build shape prediction target
        shape_pred = {
            "dominant_shape": first['dominant_shape'],
            "dominant_mass": first['dominant_mass'],
            "shape_field_mass": first['shape_field_mass'],
            "normalized_masses": first.get('normalized_masses', {}),
        }
        
        # Add composite if detected
        if first.get('composite_detected'):
            shape_pred['composite'] = first['composite']
        
        # Add confidence
        shape_pred['confidence'] = first.get('confidence', 0.0)
        
        # Create training example
        example = {
            "messages": [
                {"role": "system", "content": SHAPE_SYSTEM_PROMPT},
                {"role": "user", "content": SHAPE_USER_TEMPLATE.format(
                    contract=json.dumps(contract, ensure_ascii=False, indent=2),
                    prompt=prompt
                )},
                {"role": "assistant", "content": json.dumps(shape_pred, ensure_ascii=False)}
            ]
        }
        examples.append(example)
    
    return examples

if BUILD_DATASETS:
    shape_examples = build_shape_training_examples(shape_rows, run_contracts)
    
    # Save training examples
    shape_train_file = DATASET_DIR / "shape_critic_v1_train.jsonl"
    with open(shape_train_file, 'w') as f:
        for ex in shape_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')
    
    print(f"\nBuilt {len(shape_examples)} shape_critic examples")
    print(f"Saved to: {shape_train_file}")
    
    # Show example
    print("\nExample 1:")
    ex = shape_examples[0]
    print(f"  System: {ex['messages'][0]['content'][:80]}...")
    print(f"  User prompt: {ex['messages'][1]['content'].split('Prompt:')[1][:60]}...")
    pred = json.loads(ex['messages'][2]['content'])
    print(f"  Prediction: dominant_shape={pred['dominant_shape']}, composite={pred.get('composite', {}).get('composite', 'None')}")


Built 3 shape_critic examples
Saved to: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data\shape_critic_v1_train.jsonl

Example 1:
  System: You are the Nexus Shape Critic. Given a contract and prompt, predict the dominan...
  User prompt:  Is the contract boundary condition a filter or a gate? Defe...
  Prediction: dominant_shape=FILTER, composite=BOUNDARY


## Dataset Summary

In [17]:
# ============================================================
# DATASET SUMMARY
# ============================================================

if BUILD_DATASETS:
    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"\nv8 Corpus:")
    print(f"  Repair rows: {len(repair_rows)}")
    print(f"  Shape rows: {len(shape_rows)}")
    print(f"  Total runs: {len(runs)}")
    
    print(f"\nTraining Datasets:")
    print(f"  slot_builder_lora_v3: {len(slot_examples)} examples")
    print(f"  shape_critic_v1: {len(shape_examples)} examples")
    
    print(f"\nFiles:")
    print(f"  {DATASET_DIR / 'slot_builder_lora_v3_train.jsonl'}")
    print(f"  {DATASET_DIR / 'shape_critic_v1_train.jsonl'}")
    
    # Save manifest
    manifest = {
        "notebook": "rhi_lora_v3_training_complete",
        "model_name": MODEL_NAME,
        "v8_corpus": {
            "repair_rows": len(repair_rows),
            "shape_rows": len(shape_rows),
            "runs": len(runs)
        },
        "training_datasets": {
            "slot_builder_examples": len(slot_examples),
            "shape_critic_examples": len(shape_examples)
        },
        "hyperparameters": {
            "slot_builder": {
                "rank": SLOT_LORA_RANK,
                "alpha": SLOT_LORA_ALPHA,
                "target_modules": SLOT_TARGET_MODULES,
                "epochs": SLOT_EPOCHS,
                "lr": SLOT_LR
            },
            "shape_critic": {
                "rank": SHAPE_LORA_RANK,
                "alpha": SHAPE_LORA_ALPHA,
                "target_modules": SHAPE_TARGET_MODULES,
                "epochs": SHAPE_EPOCHS,
                "lr": SHAPE_LR
            }
        },
        "training_enabled": RUN_TRAINING
    }
    
    manifest_file = DATASET_DIR / "training_manifest.json"
    with open(manifest_file, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"\nManifest saved to: {manifest_file}")

DATASET SUMMARY

v8 Corpus:
  Repair rows: 78
  Shape rows: 12
  Total runs: 16

Training Datasets:
  slot_builder_lora_v3: 16 examples
  shape_critic_v1: 3 examples

Files:
  D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data\slot_builder_lora_v3_train.jsonl
  D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data\shape_critic_v1_train.jsonl

Manifest saved to: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_lora_v3_training_data\training_manifest.json


## Load Base Model and Tokenizer

In [18]:
# ============================================================
# LOAD BASE MODEL
# ============================================================

if RUN_TRAINING:
    print("Loading base model and tokenizer...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Configure quantization if enabled
    if USE_4BIT:
        print("Using 4-bit quantization...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        base_model = prepare_model_for_kbit_training(base_model)
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    
    print(f"Model loaded: {MODEL_NAME}")
    print(f"Device: {base_model.device}")
else:
    print("\nRUN_TRAINING=False, skipping model loading.")
    print("Set RUN_TRAINING=True to begin training.")

Loading base model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


## Train slot_builder_lora_v3

In [19]:
# ============================================================
# TRAIN SLOT_BUILDER_LORA_V3
# ============================================================

if RUN_TRAINING:
    print("=" * 60)
    print("TRAINING SLOT_BUILDER_LORA_V3")
    print("=" * 60)
    
    # Load training data
    slot_train_file = DATASET_DIR / "slot_builder_lora_v3_train.jsonl"
    with open(slot_train_file) as f:
        slot_train_data = [json.loads(line) for line in f]
    
    print(f"\nLoaded {len(slot_train_data)} training examples")
    
    # Format for training
    def format_slot_example(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}
    
    slot_dataset = Dataset.from_list(slot_train_data).map(format_slot_example)
    
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )
    
    slot_tokenized = slot_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    
    # LoRA configuration
    slot_lora_config = LoraConfig(
        r=SLOT_LORA_RANK,
        lora_alpha=SLOT_LORA_ALPHA,
        target_modules=SLOT_TARGET_MODULES,
        lora_dropout=SLOT_LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    # Create PEFT model
    slot_model = get_peft_model(base_model, slot_lora_config)
    slot_model.print_trainable_parameters()
    
    # Training arguments
    slot_training_args = TrainingArguments(
        output_dir=str(SLOT_OUTPUT_DIR),
        num_train_epochs=SLOT_EPOCHS,
        per_device_train_batch_size=SLOT_BATCH_SIZE,
        gradient_accumulation_steps=SLOT_GRAD_ACCUM,
        learning_rate=SLOT_LR,
        fp16=True,
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_torch",
        warmup_ratio=0.1,
        report_to="none",
    )
    
    # Trainer
    slot_trainer = Trainer(
        model=slot_model,
        args=slot_training_args,
        train_dataset=slot_tokenized,
    )
    
    # Train
    print("\nStarting training...")
    slot_trainer.train()
    
    # Save
    print(f"\nSaving adapter to: {SLOT_OUTPUT_DIR}")
    slot_model.save_pretrained(str(SLOT_OUTPUT_DIR))
    tokenizer.save_pretrained(str(SLOT_OUTPUT_DIR))
    
    print("\nslot_builder_lora_v3 training complete!")
    
    # Clean up
    del slot_model
    del slot_trainer
    torch.cuda.empty_cache()
else:
    print("\nRUN_TRAINING=False, skipping slot_builder training.")

TRAINING SLOT_BUILDER_LORA_V3

Loaded 16 training examples


Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

W0507 04:33:54.386000 51476 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410

Starting training...


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

## Train shape_critic_v1

In [ ]:
# ============================================================
# TRAIN SHAPE_CRITIC_V1
# ============================================================

if RUN_TRAINING:
    print("=" * 60)
    print("TRAINING SHAPE_CRITIC_V1")
    print("=" * 60)
    
    # Load training data
    shape_train_file = DATASET_DIR / "shape_critic_v1_train.jsonl"
    with open(shape_train_file) as f:
        shape_train_data = [json.loads(line) for line in f]
    
    print(f"\nLoaded {len(shape_train_data)} training examples")
    
    # Format for training
    def format_shape_example(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}
    
    shape_dataset = Dataset.from_list(shape_train_data).map(format_shape_example)
    
    # Tokenize
    shape_tokenized = shape_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    
    # LoRA configuration
    shape_lora_config = LoraConfig(
        r=SHAPE_LORA_RANK,
        lora_alpha=SHAPE_LORA_ALPHA,
        target_modules=SHAPE_TARGET_MODULES,
        lora_dropout=SHAPE_LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    # Create PEFT model (reload base model if needed)
    if USE_4BIT:
        base_model_fresh = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        base_model_fresh = prepare_model_for_kbit_training(base_model_fresh)
    else:
        base_model_fresh = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
    
    shape_model = get_peft_model(base_model_fresh, shape_lora_config)
    shape_model.print_trainable_parameters()
    
    # Training arguments
    shape_training_args = TrainingArguments(
        output_dir=str(SHAPE_OUTPUT_DIR),
        num_train_epochs=SHAPE_EPOCHS,
        per_device_train_batch_size=SHAPE_BATCH_SIZE,
        gradient_accumulation_steps=SHAPE_GRAD_ACCUM,
        learning_rate=SHAPE_LR,
        fp16=True,
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_torch",
        warmup_ratio=0.2,  # More warmup for small dataset
        report_to="none",
    )
    
    # Trainer
    shape_trainer = Trainer(
        model=shape_model,
        args=shape_training_args,
        train_dataset=shape_tokenized,
    )
    
    # Train
    print("\nStarting training...")
    shape_trainer.train()
    
    # Save
    print(f"\nSaving adapter to: {SHAPE_OUTPUT_DIR}")
    shape_model.save_pretrained(str(SHAPE_OUTPUT_DIR))
    tokenizer.save_pretrained(str(SHAPE_OUTPUT_DIR))
    
    print("\nshape_critic_v1 training complete!")
    
    # Clean up
    del shape_model
    del shape_trainer
    del base_model_fresh
    torch.cuda.empty_cache()
else:
    print("\nRUN_TRAINING=False, skipping shape_critic training.")

## Training Complete

In [ ]:
# ============================================================
# TRAINING SUMMARY
# ============================================================

if RUN_TRAINING:
    print("=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    
    print("\nAdapters saved to:")
    print(f"  slot_builder_lora_v3: {SLOT_OUTPUT_DIR}")
    print(f"  shape_critic_v1: {SHAPE_OUTPUT_DIR}")
    
    print("\nNext steps:")
    print("  1. Validate slot_builder_lora_v3 on held-out prompts")
    print("  2. Validate shape_critic_v1 on held-out prompts")
    print("  3. Integrate both adapters into v9 runtime")
    print("  4. Test v9 shape-guided synthesis")
    print("  5. Collect v9 corpus (20+ runs)")
    print("  6. Retrain v4 adapters on expanded corpus")
    
    print("\nv9 runtime shape:")
    print("  Q → C_v3 → K*_shape_critic → A_shape_guided → audit → Ψ/Ω")
    
    # Verify files exist
    slot_adapter = SLOT_OUTPUT_DIR / "adapter_model.safetensors"
    shape_adapter = SHAPE_OUTPUT_DIR / "adapter_model.safetensors"
    
    print("\nAdapter files:")
    print(f"  slot_builder adapter exists: {slot_adapter.exists()}")
    print(f"  shape_critic adapter exists: {shape_adapter.exists()}")
    
else:
    print("\n" + "=" * 60)
    print("READY TO TRAIN")
    print("=" * 60)
    print("\nDatasets built and ready.")
    print("\nTo begin training:")
    print("  1. Set RUN_TRAINING = True in the config cell")
    print("  2. Restart kernel and run all cells")
    print("\nExpected training time:")
    print("  slot_builder_lora_v3: ~20-30 minutes")
    print("  shape_critic_v1: ~15-20 minutes")
    print("  Total: ~45 minutes on GPU")